# MangoFS-BD: few-shot recognition of Bangladeshi mango cultivars

Few-shot recognition of unseen mango cultivars on a leakage-controlled, cultivar-disjoint benchmark; Prototypical Networks, metric-learning baselines, KAN heads and a KAN metric, and ordinary fine-tuning.

**Benchmark:** 24 cultivars and 11,613 images merged from MangoImageBD, MangoClassify-12 and Mangifera2012.

**How to run on Kaggle:** first build the image cache in a CPU notebook with internet on and the MangoClassify-12 dataset attached (`import prep_data; prep_data.main()`, section 2.2). Then, in this notebook, *Add input* → that notebook's output and a dataset containing `benchmark/groups.csv` from the repository; Accelerator **GPU T4 ×2**; Internet **on**. Section 2 writes the library to `src/`; sections 3–6 run it.

This notebook is generated from the repository by `tools/make_notebook.py`. Edit `src/` instead.

## 1. Setup

In [ ]:
import os, sys, glob
os.makedirs('src', exist_ok=True)
sys.path.insert(0, os.path.abspath('src'))
import torch
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# 2. Library code

## 2.1 Preprocessing
Trim padding bars, find the fruit (largest compact, saturated blob), square crop around it, resize to 288 px; DCT perceptual hash.

In [ ]:
%%writefile src/preprocess.py
"""Image preprocessing shared by the local check and the Kaggle prep kernel.

Every raw image goes through the same steps so that no dataset gets its own
"signature" a model could shortcut on:
  1. trim_borders  - removes uniform padding strips (MangoImageBD pads many
                     photos to 504x1120 with flat white/blue bars).
  2. fruit_box     - finds the fruit as the largest saturated blob (mangoes are
                     green/yellow/orange; backgrounds are mostly white/grey).
  3. square crop   - square crop around the fruit with a margin, falling back
                     to a centre crop when no plausible blob is found.
  4. resize        - to SIZE x SIZE.
A 64-bit DCT perceptual hash is also computed for duplicate / near-duplicate
grouping (see grouping.py).
"""
import numpy as np
import cv2

SIZE = 288


def trim_borders(a, tol=12):
    """Drop edge rows/cols that are a single flat colour (padding)."""
    a16 = a.astype(np.int16)
    rdev = np.abs(a16 - np.median(a16, axis=1, keepdims=True)).max(axis=(1, 2))
    cdev = np.abs(a16 - np.median(a16, axis=0, keepdims=True)).max(axis=(0, 2))
    r = np.where(rdev > tol)[0]
    c = np.where(cdev > tol)[0]
    if len(r) < 32 or len(c) < 32:
        return a
    return a[r[0]:r[-1] + 1, c[0]:c[-1] + 1]


def fruit_box(a):
    """Bounding box (x0, y0, x1, y1) of the largest saturated blob, or None."""
    h, w = a.shape[:2]
    scale = 256.0 / max(h, w)
    small = cv2.resize(a, (max(1, int(w * scale)), max(1, int(h * scale))),
                       interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    hue, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    # mango hues: orange..green (OpenCV hue 5..95), reasonably saturated
    mask = ((s > 60) & (v > 40) & (hue >= 5) & (hue <= 95)).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    H, W = mask.shape
    best, best_score = None, 0.0
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        frac = area / float(mask.size)
        if frac < 0.02 or frac > 0.9:
            continue
        # a fruit is a compact blob; table edges / wooden planks are long
        # strips that run into the image border
        fill = area / float(bw * bh)
        sides = int(x <= 1) + int(y <= 1) + int(x + bw >= W - 1) + int(y + bh >= H - 1)
        elong = max(bw, bh) / float(min(bw, bh))
        # mango skin is strongly saturated; wood / clutter much less so
        sat = float(s[lab == i].mean()) / 255.0
        score = area * fill * sat ** 2 * (0.3 ** sides) / max(1.0, elong - 1.0)
        if score > best_score:
            best, best_score = (x, y, bw, bh), score
    if best is None:
        return None
    x, y, bw, bh = best
    return (x / scale, y / scale, (x + bw) / scale, (y + bh) / scale)


def square_crop(a, box, margin=0.12):
    h, w = a.shape[:2]
    if box is None:
        side = min(h, w)
        cx, cy = w / 2, h / 2
    else:
        x0, y0, x1, y1 = box
        side = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
        side = min(max(side, 0.35 * min(h, w)), min(h, w))
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    half = side / 2
    cx = min(max(cx, half), w - half)
    cy = min(max(cy, half), h - half)
    x0, y0 = int(round(cx - half)), int(round(cy - half))
    s = int(round(side))
    return a[y0:y0 + s, x0:x0 + s]


def phash(a):
    """64-bit DCT perceptual hash as a python int."""
    g = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(g)[:8, :8].flatten()
    bits = d > np.median(d[1:])
    return int("".join("1" if b else "0" for b in bits), 2)


def process(a, size=SIZE):
    """RGB uint8 array -> (size x size RGB uint8, found_fruit, phash)."""
    a = trim_borders(a)
    box = fruit_box(a)
    out = cv2.resize(square_crop(a, box), (size, size), interpolation=cv2.INTER_AREA)
    return out, box is not None, phash(out)


def load_rgb(path, max_side=1600):
    """Read with EXIF orientation applied, downscaled early for speed."""
    from PIL import Image, ImageOps
    im = Image.open(path)
    im.draft("RGB", (max_side, max_side))  # fast JPEG DCT-domain downscale
    im = ImageOps.exif_transpose(im).convert("RGB")
    if max(im.size) > max_side:
        im.thumbnail((max_side, max_side), Image.BILINEAR)
    return np.asarray(im)

## 2.2 Building the unified image cache
Downloads MangoImageBD and Mangifera2012 from Mendeley, reads MangoClassify-12 from Kaggle, harmonises cultivar names (24 cultivars) and writes `crop.npy`, `full.npy`, `meta.csv`.

In [ ]:
%%writefile src/prep_data.py
"""Build the unified MangoFS-BD image cache from the three public datasets.

Runs on Kaggle (internet on). Output in OUT_DIR:
  crop.npy   (N, S, S, 3) uint8   fruit-centred square crops
  full.npy   (N, S, S, 3) uint8   trimmed full frame, centre square crop
  meta.csv   one row per image (dataset, raw label, canonical cultivar,
             path, exif time/camera, perceptual hash, crop flag)
"""
import csv
import glob
import os
import re
import subprocess
import sys
import zipfile
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import cv2

from preprocess import SIZE, load_rgb, process, trim_borders, square_crop

MENDELEY = {
    "MangoImageBD": "https://data.mendeley.com/public-files/datasets/hp2cdckpdr/files/"
                    "28bb7356-9476-4b74-82b3-b445232e254c/file_downloaded",
    "Mangifera2012": "https://data.mendeley.com/public-files/datasets/w5jg84txj8/files/"
                     "c34ac544-aea9-451a-bce9-953d1ca645a9/file_downloaded",
}
D2_ROOT_CANDIDATES = glob.glob("/kaggle/input/**/MangoClassify-12", recursive=True)

# Raw folder name -> canonical cultivar.  Same cultivar under different
# spellings across datasets is merged so that a "novel" cultivar is novel
# in every dataset at once.
CANON = {
    "amrapali": "Amrapali",
    "ashshina classic": "AshshinaClassic",
    "ashshina zhinuk": "AshshinaZhinuk",
    "banana mango": "Banana", "banana": "Banana",
    "bari-4": "Bari4", "bari 4": "Bari4", "bari4": "Bari4",
    "bari-7": "Bari7", "bari 7": "Bari7",
    "bari-11": "Bari11",
    "fazli classic": "Fazli", "fazli": "Fazli", "fazlee": "Fazli",
    "fazli shurmai": "FazliShurmai",
    "gourmoti": "Gourmoti",
    "harivanga": "Harivanga", "haribhanga": "Harivanga",
    "himsagor": "Himsagar", "himsagar": "Himsagar",
    "katimon": "Katimon",
    "langra": "Langra",
    "kanchon langra": "KanchonLangra",
    "rupali": "Rupali",
    "shada": "Shada",
    "gobindobhog": "GobindoBhog",
    "gopalbhog": "GopalBhog",
    "khrishapat": "Khirsapat",
    "ranibhog": "RaniBhog",
    "sundari": "Sundari",
    "mollika": "Mollika",
    "nilambori": "Nilambori",
}


def canon(name):
    # Mangifera2012 folders carry their image count, e.g. "Amrapali-252"
    k = re.sub(r"-\d{3}$", "", name.strip()).lower().replace("_", " ")
    if k not in CANON:
        raise KeyError(f"unknown cultivar folder {name!r}")
    return CANON[k]


def fetch(name, url, work):
    z = os.path.join(work, name + ".zip")
    if not os.path.exists(z):
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", z, url], check=True)
    d = os.path.join(work, name)
    if not os.path.isdir(d):
        with zipfile.ZipFile(z) as f:
            f.extractall(d)
        os.remove(z)
    return d


def list_images(root):
    exts = (".jpg", ".jpeg", ".png")
    out = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(exts) and not f.startswith("."):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


def exif_info(path):
    from PIL import Image
    try:
        ex = Image.open(path).getexif()
        sub = ex.get_ifd(0x8769)
        t = sub.get(36867) or ex.get(306) or ""
        return str(t), str(ex.get(272) or "").strip()
    except Exception:
        return "", ""


def work_one(args):
    ds, label, path = args
    try:
        a = load_rgb(path)
    except Exception:  # corrupt file
        return None
    crop, found, h = process(a)
    full = cv2.resize(square_crop(trim_borders(a), None), (SIZE, SIZE),
                      interpolation=cv2.INTER_AREA)
    t, cam = exif_info(path)
    return dict(dataset=ds, raw_label=label, cultivar=canon(label),
                path=path, exif_time=t, camera=cam, phash=f"{h:016x}",
                fruit_found=int(found), h=a.shape[0], w=a.shape[1]), crop, full


def main(out_dir="/kaggle/working", work="/tmp/mango"):
    os.makedirs(work, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)
    jobs = []
    d1 = fetch("MangoImageBD", MENDELEY["MangoImageBD"], work)
    for p in list_images(d1):
        jobs.append(("MangoImageBD", os.path.basename(os.path.dirname(p)), p))
    d3 = fetch("Mangifera2012", MENDELEY["Mangifera2012"], work)
    for p in list_images(d3):
        jobs.append(("Mangifera2012", os.path.basename(os.path.dirname(p)), p))
    if not D2_ROOT_CANDIDATES:
        sys.exit("MangoClassify-12 input not attached")
    for p in list_images(D2_ROOT_CANDIDATES[0]):
        jobs.append(("MangoClassify12", os.path.basename(os.path.dirname(p)), p))
    labels = sorted({(j[0], j[1]) for j in jobs})
    print(len(jobs), "images;", labels, flush=True)
    for ds, lab in labels:
        canon(lab)  # fail fast on unknown folder names

    rows, crops, fulls = [], [], []
    with ProcessPoolExecutor(os.cpu_count()) as ex:
        for i, r in enumerate(ex.map(work_one, jobs, chunksize=16)):
            if r is None:
                print("skip corrupt", jobs[i][2])
                continue
            m, c, f = r
            m["idx"] = len(rows)
            rows.append(m)
            crops.append(c)
            fulls.append(f)
            if i % 1000 == 0:
                print(i, flush=True)
    np.save(os.path.join(out_dir, "crop.npy"), np.stack(crops))
    np.save(os.path.join(out_dir, "full.npy"), np.stack(fulls))
    with open(os.path.join(out_dir, "meta.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print("done", len(rows), "fruit found:", sum(r["fruit_found"] for r in rows))
    # contact sheet per dataset for visual QA of the crops
    rng = np.random.default_rng(0)
    for ds in sorted({r["dataset"] for r in rows}):
        ids = [r["idx"] for r in rows if r["dataset"] == ds]
        pick = rng.choice(ids, size=min(60, len(ids)), replace=False)
        tiles = [cv2.resize(crops[i], (128, 128)) for i in pick]
        tiles += [np.zeros_like(tiles[0])] * (-len(tiles) % 10)
        sheet = np.vstack([np.hstack(tiles[k:k + 10]) for k in range(0, len(tiles), 10)])
        cv2.imwrite(os.path.join(out_dir, f"sheet_{ds}.jpg"), sheet[..., ::-1])


if __name__ == "__main__":
    main()

## 2.2b Background-removed variant
Segments every fruit crop with U$^2$-Net (rembg) and composites it on a white background (`seg.npy`).

In [ ]:
%%writefile src/prep_seg.py
"""Background-removed variant of the image cache (seg.npy).

    import prep_seg; prep_seg.main()

The fruit crops of crop.npy are segmented with the U^2-Net salient-object model
(rembg, model "u2net") and composited on a uniform white background, so that the
background of the source dataset no longer differs between sources. The same
processing is applied to every image. If the mask covers less than 2% of the crop
(nothing found), the original crop is kept and flagged.

Writes to the working directory:
  seg.npy        uint8 (N, 288, 288, 3), aligned with meta.csv
  meta.csv       copy of the cache metadata (so the variant can be found)
  seg_stats.csv  idx, mask_frac (mean alpha), fallback (1 = original crop kept)
"""
import glob
import os
import shutil
import subprocess
import sys
import time

INPUT = os.environ.get("MANGOFS_INPUT", "/kaggle/input")
WORK = os.environ.get("MANGOFS_WORK", "/kaggle/working")


def composite(img, alpha):
    a = alpha.astype("float32")[..., None] / 255.0
    return (img.astype("float32") * a + 255.0 * (1.0 - a)).round().astype("uint8")


def main(model="u2net", min_frac=0.02, limit=None):
    try:
        import rembg  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rembg[cpu]"], check=True)
    import numpy as np
    import pandas as pd
    from PIL import Image
    from rembg import new_session, remove

    root = os.path.dirname(glob.glob(f"{INPUT}/**/crop.npy", recursive=True)[0])
    crop = np.load(os.path.join(root, "crop.npy"), mmap_mode="r")
    shutil.copy(os.path.join(root, "meta.csv"), os.path.join(WORK, "meta.csv"))
    os.environ.setdefault("U2NET_HOME", "/tmp/u2net")  # keep the model out of the outputs
    sess = new_session(model)
    n = len(crop) if limit is None else limit  # limit: quick test on the first images
    out = np.lib.format.open_memmap(os.path.join(WORK, "seg.npy"), mode="w+",
                                    dtype=np.uint8, shape=(n,) + crop.shape[1:])
    rows, t0 = [], time.time()
    for i in range(n):
        img = np.asarray(crop[i])
        alpha = np.asarray(remove(Image.fromarray(img), session=sess, only_mask=True))
        frac = float(alpha.mean() / 255.0)
        fallback = int(frac < min_frac)
        out[i] = img if fallback else composite(img, alpha)
        rows.append({"idx": i, "mask_frac": round(frac, 4), "fallback": fallback})
        if i % 500 == 0:
            print(f"{i}/{n} {time.time() - t0:.0f}s", flush=True)
    out.flush()
    s = pd.DataFrame(rows)
    s.to_csv(os.path.join(WORK, "seg_stats.csv"), index=False)
    print("done:", len(s), "images, fallback", int(s.fallback.sum()),
          "median mask fraction", float(s.mask_frac.median()), flush=True)

## 2.3 Data: capture groups, protocol splits, samplers, GPU augmentation
Capture groups keep multi-view shots / near-duplicates on one side of every split; episodes are sampled so support and query never share a group.

In [ ]:
%%writefile src/data.py
"""Image cache, leakage-safe grouping, protocol splits, samplers, GPU augmentation."""
import os
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

DATASETS = ["MangoImageBD", "MangoClassify12", "Mangifera2012"]


class Cache:
    """uint8 image array (N,S,S,3) kept in RAM + metadata dataframe."""

    def __init__(self, root, variant="crop", groups_path=None, in_memory=True):
        self.root = root
        self.meta = pd.read_csv(os.path.join(root, "meta.csv"))
        gpath = groups_path or os.path.join(root, "groups.csv")
        if os.path.exists(gpath):
            g = pd.read_csv(gpath)
            self.meta = self.meta.merge(g[["idx", "group", "drop"]], on="idx", how="left")
        self.images = np.load(os.path.join(root, f"{variant}.npy"), mmap_mode="r")
        if in_memory:  # training reads random batches; keep the array in RAM
            self.images = np.ascontiguousarray(self.images)
        assert len(self.images) == len(self.meta)

    def batch(self, idx, device):
        x = torch.from_numpy(self.images[np.asarray(idx)]).to(device, non_blocking=True)
        return x.permute(0, 3, 1, 2).float().div_(255)


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[max(a, b)] = min(a, b)


def _parse_time(t):
    try:
        return datetime.strptime(str(t)[:19], "%Y:%m:%d %H:%M:%S").timestamp()
    except Exception:
        return None


def build_groups(meta, feats=None, seq_gap_s=10, dup_cos=0.97, xlabel_dup_cos=0.97, phash_thr=2):
    """Capture-group ids used to keep related photos on one side of every split.

    Two images share a group (within one dataset and cultivar) if
      * they belong to the same capture sequence: same camera, consecutive
        EXIF timestamps <= seq_gap_s apart (multi-view shots of a fruit), or
      * they are near-duplicates: complete-linkage clusters of DINOv2
        embeddings with every pairwise cosine >= dup_cos (complete linkage
        avoids the chaining that single linkage produces among same-cultivar
        fruits, which are all fairly similar), or
      * their perceptual hashes differ by <= phash_thr bits AND (if features
        are given) cosine >= dup_cos (re-encoded copies).
    An image whose near-copy (cosine >= xlabel_dup_cos, or identical pHash)
    carries a *different* cultivar label is marked drop=1 (label conflict).

    Note: pHash alone is unreliable here - centred fruit on plain backgrounds
    gives many low-Hamming pairs across cultivars (see docs/grouping.md).
    """
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    n = len(meta)
    uf = UnionFind(n)
    drop = np.zeros(n, dtype=int)
    stats = {"seq_links": 0, "dup_links": 0, "phash_links": 0, "cross_label_dups": 0,
             "cross_dataset_dups": 0}
    f = None if feats is None else F.normalize(torch.as_tensor(feats).float(), dim=1)
    ds = meta["dataset"].to_numpy()
    cv = meta["cultivar"].to_numpy()

    # 1) capture sequences
    t = meta["exif_time"].map(_parse_time)
    key = meta["dataset"] + "|" + meta["cultivar"] + "|" + meta["camera"].fillna("")
    df = pd.DataFrame({"t": t, "key": key, "i": np.arange(n)}).dropna(subset=["t"])
    for _, g in df.groupby("key"):
        g = g.sort_values("t")
        ts, ids = g["t"].to_numpy(), g["i"].to_numpy()
        for k in range(1, len(ts)):
            if ts[k] - ts[k - 1] <= seq_gap_s:
                uf.union(int(ids[k - 1]), int(ids[k]))
                stats["seq_links"] += 1

    # 2) near-duplicate clusters (complete linkage) per dataset x cultivar
    if f is not None:
        key2 = meta["dataset"] + "|" + meta["cultivar"]
        for _, ids in meta.groupby(key2).indices.items():
            if len(ids) < 2:
                continue
            D = (1 - f[ids] @ f[ids].t()).clamp_min(0).numpy().astype(np.float64)
            np.fill_diagonal(D, 0)
            lab = fcluster(linkage(squareform(D, checks=False), "complete"), 1 - dup_cos, "distance")
            for c in np.unique(lab):
                mem = ids[lab == c]
                for j in mem[1:]:
                    uf.union(int(mem[0]), int(j))
                    stats["dup_links"] += 1
        # label conflicts: near-identical images under different cultivars
        for s0 in range(0, n, 2048):
            sim = f[s0:s0 + 2048] @ f.t()
            ii, jj = (sim >= xlabel_dup_cos).nonzero(as_tuple=True)
            for i, j in zip((ii + s0).tolist(), jj.tolist()):
                if i < j and cv[i] != cv[j]:
                    drop[i] = drop[j] = 1
                    stats["cross_label_dups"] += 1
                if i < j and ds[i] != ds[j]:
                    stats["cross_dataset_dups"] += 1

    # 3) re-encoded copies via pHash
    h = np.array([int(x, 16) for x in meta["phash"]], dtype=np.uint64)
    B = torch.from_numpy(np.unpackbits(h.view(np.uint8).reshape(n, 8), axis=1).astype(np.float32))
    for s0 in range(0, n, 2048):
        ham = B[s0:s0 + 2048] @ (1 - B).t() + (1 - B[s0:s0 + 2048]) @ B.t()
        ii, jj = (ham <= phash_thr).nonzero(as_tuple=True)
        for i, j in zip((ii + s0).tolist(), jj.tolist()):
            if i >= j:
                continue
            if f is not None and float(f[i] @ f[j]) < dup_cos:
                continue
            if cv[i] != cv[j]:
                if f is None and ham[i - s0, j] == 0:
                    drop[i] = drop[j] = 1
                    stats["cross_label_dups"] += 1
                continue
            if ds[i] == ds[j]:
                uf.union(i, j)
                stats["phash_links"] += 1
    roots = np.array([uf.find(i) for i in range(n)])
    _, group = np.unique(roots, return_inverse=True)
    return group, drop, stats


def cultivar_folds(meta, n_folds=3, seed=0):
    """Assign every cultivar to exactly one novel fold.

    Cultivars are sorted by image count and dealt round-robin (snake order)
    so every fold gets a similar mix of large and small cultivars."""
    counts = meta.groupby("cultivar").size().sort_values(ascending=False)
    rng = np.random.default_rng(seed)
    names = list(counts.index)
    folds = [[] for _ in range(n_folds)]
    for r in range(0, len(names), n_folds):
        chunk = names[r:r + n_folds]
        order = rng.permutation(n_folds) if r // n_folds % 2 == 0 else rng.permutation(n_folds)[::-1]
        for name, f in zip(chunk, order):
            folds[f].append(name)
    return [sorted(f) for f in folds]


def protocol_split(meta, protocol, fold, seed=0):
    """Return (train_idx, test_idx, info) for a protocol/fold.

    unified : cultivar-disjoint 3-fold CV over the merged 24-cultivar pool;
              train = all images of base cultivars, test = novel cultivars.
    lodo    : leave-one-dataset-out; train = the two other datasets, test =
              every class of the held-out dataset.  Test classes are tagged
              'seen' (cultivar present in train under another domain) or
              'novel'.
    """
    m = meta[meta["drop"] == 0] if "drop" in meta else meta
    if protocol == "unified":
        folds = cultivar_folds(m, 3, seed)
        novel = set(folds[fold])
        test = m[m.cultivar.isin(novel)]
        train = m[~m.cultivar.isin(novel)]
        info = {"novel": sorted(novel), "base": sorted(set(train.cultivar))}
    elif protocol == "lodo":
        held = DATASETS[fold]
        test = m[m.dataset == held]
        train = m[m.dataset != held]
        seen = set(train.cultivar)
        info = {"held_out": held,
                "seen": sorted(set(test.cultivar) & seen),
                "novel": sorted(set(test.cultivar) - seen),
                "base": sorted(seen)}
    else:
        raise ValueError(protocol)
    return train.idx.to_numpy(), test.idx.to_numpy(), info


class PKSampler:
    """P classes x M images per batch, class-balanced (images drawn uniformly within a class)."""

    def __init__(self, labels, P, M, seed=0):
        self.rng = np.random.default_rng(seed)
        self.labels = np.asarray(labels)
        self.classes = np.unique(self.labels)
        self.by_class = {c: np.where(self.labels == c)[0] for c in self.classes}
        self.P, self.M = min(P, len(self.classes)), M

    def sample(self):
        cls = self.rng.choice(self.classes, self.P, replace=False)
        out = []
        for c in cls:
            pool = self.by_class[c]
            out.append(self.rng.choice(pool, self.M, replace=len(pool) < self.M))
        return np.concatenate(out)  # positions into labels array, class-contiguous


def sample_episode(rng, labels, groups, classes, n_way, k_shot, n_query, group_disjoint=True,
                   domains=None, support_domain=None, query_domain=None):
    """Sample an N-way K-shot episode; returns (support_pos, ys, query_pos, yq, classes).

    group_disjoint: support and query images never share a capture group, so a
    query can never be a near-copy / another view of a support fruit.
    """
    cls = rng.choice(classes, n_way, replace=False)
    sp, ys, qp, yq = [], [], [], []
    for i, c in enumerate(cls):
        pos = np.where(labels == c)[0]
        if support_domain is not None:
            s_pool = pos[domains[pos] == support_domain]
            q_pool = pos[domains[pos] == query_domain]
        else:
            s_pool = q_pool = pos
        if group_disjoint:
            # draw support groups at random until K images are collected,
            # query from the remaining groups
            g = groups[s_pool]
            ug = rng.permutation(np.unique(g))
            chosen, s = set(), []
            for gg in ug:
                members = s_pool[g == gg]
                s.extend(rng.permutation(members)[: k_shot - len(s)])
                chosen.add(gg)
                if len(s) >= k_shot:
                    break
            q_pool = q_pool[~np.isin(groups[q_pool], list(chosen))]
            s = np.array(s)
        else:
            s = rng.choice(s_pool, k_shot, replace=False)
            q_pool = np.setdiff1d(q_pool, s)
        q = rng.choice(q_pool, min(n_query, len(q_pool)), replace=False)
        sp.append(s); qp.append(q)
        ys += [i] * len(s); yq += [i] * len(q)
    return np.concatenate(sp), np.array(ys), np.concatenate(qp), np.array(yq), cls


def eligible_classes(labels, groups, k_shot, n_query, min_query=5):
    """Classes that can supply k_shot support + min_query group-disjoint queries."""
    out = []
    for c in np.unique(labels):
        _, cnt = np.unique(groups[labels == c], return_counts=True)
        cnt = np.sort(cnt)[::-1]
        # worst case: the support is drawn from the largest groups
        n_sup_groups = int(np.searchsorted(np.cumsum(cnt), k_shot)) + 1
        if cnt.sum() - cnt[:n_sup_groups].sum() >= min_query:
            out.append(c)
    return np.array(out)


def gpu_augment(x, out_res, scale=(0.45, 1.0), max_rot=25, jitter=0.15, train=True):
    """Per-sample random resized crop + flip + rotation + mild colour jitter.

    x: (B,3,S,S) float in [0,1] on GPU.  Hue is deliberately left untouched:
    skin colour is a genuine cultivar cue.
    """
    B = x.shape[0]
    dev = x.device
    if not train:
        return F.interpolate(x, size=(out_res, out_res), mode="bilinear",
                             antialias=True, align_corners=False)
    s = torch.empty(B, device=dev).uniform_(*scale).sqrt()          # side fraction
    ar = torch.exp(torch.empty(B, device=dev).uniform_(-0.2, 0.2))   # aspect jitter
    sx, sy = s * ar.sqrt(), s / ar.sqrt()
    sx, sy = sx.clamp(max=1), sy.clamp(max=1)
    tx = (torch.rand(B, device=dev) * 2 - 1) * (1 - sx)
    ty = (torch.rand(B, device=dev) * 2 - 1) * (1 - sy)
    ang = (torch.rand(B, device=dev) * 2 - 1) * max_rot * np.pi / 180
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([
        torch.stack([sx * cos * flip, -sy * sin, tx], 1),
        torch.stack([sx * sin * flip, sy * cos, ty], 1)], 1)
    # downscale first (antialias) so the sampled crop is not aliased
    if x.shape[-1] > out_res * 1.25:
        x = F.interpolate(x, size=(int(out_res * 1.25),) * 2, mode="bilinear",
                          antialias=True, align_corners=False)
    grid = F.affine_grid(theta, (B, 3, out_res, out_res), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if jitter > 0:
        b = 1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * jitter
        c = 1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * jitter
        sat = 1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * jitter
        mean = x.mean((1, 2, 3), keepdim=True)
        x = (x - mean) * c + mean
        x = x * b
        gray = x.mean(1, keepdim=True)
        x = (x - gray) * sat + gray
    return x.clamp(0, 1)

## 2.4 Models
Backbones (Conv-4, ResNet-18/50, DenseNet-121, DINOv2 ViT-S/14 and ViT-B/14), MLP and KAN projection heads, and the proposed **KAN metric** $d(q,c)=\lVert q-c\rVert^2+\sum_j \varphi_j(|q_j-c_j|)$.

In [ ]:
%%writefile src/models.py
"""Backbones, projection heads (identity / MLP / KAN) and the embedding model.

Embedding model:  image -> backbone features f (d_f) -> head g -> z (d_z)
The head is the only part that changes between the "MLP" and "KAN" arms of
the study, so every other factor (backbone, objective, data, schedule) is
held fixed when the two are compared.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# Kolmogorov-Arnold layer (B-spline KAN, Liu et al. 2024; vectorised as in
# the "efficient-kan" formulation).  Each edge i->j carries
#     phi_ij(x) = w_b * silu(x) + w_s * sum_k c_ijk B_k(x)
# with cubic B-splines on a uniform grid.
class KANLinear(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3,
                 grid_range=(-2.0, 2.0), scale_noise=0.1):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.grid_size, self.spline_order = grid_size, spline_order
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]
        self.register_buffer("grid", grid.expand(in_features, -1).contiguous())
        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.spline_weight = nn.Parameter(
            torch.empty(out_features, in_features, grid_size + spline_order))
        self.spline_scaler = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5))
        with torch.no_grad():
            self.spline_weight.uniform_(-scale_noise / grid_size, scale_noise / grid_size)
        nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5))

    def b_splines(self, x):
        # x: (B, in) -> (B, in, grid_size + spline_order)
        g = self.grid
        x = x.unsqueeze(-1)
        bases = ((x >= g[:, :-1]) & (x < g[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = ((x - g[:, :-(k + 1)]) / (g[:, k:-1] - g[:, :-(k + 1)]) * bases[..., :-1]
                     + (g[:, k + 1:] - x) / (g[:, k + 1:] - g[:, 1:-k]) * bases[..., 1:])
        return bases

    def forward(self, x):
        x = x.float()  # spline recursion is not fp16-safe
        base = F.linear(F.silu(x), self.base_weight)
        w = self.spline_weight * self.spline_scaler.unsqueeze(-1)
        spline = F.linear(self.b_splines(x).flatten(1), w.flatten(1))
        return base + spline


class KANHead(nn.Module):
    """LayerNorm keeps inputs inside the spline grid; two KAN layers."""

    def __init__(self, d_in, d_hidden=256, d_out=128, grid_size=5):
        super().__init__()
        self.norm = nn.LayerNorm(d_in)
        self.l1 = KANLinear(d_in, d_hidden, grid_size=grid_size)
        self.norm2 = nn.LayerNorm(d_hidden)
        self.l2 = KANLinear(d_hidden, d_out, grid_size=grid_size)

    def forward(self, x):
        return self.l2(self.norm2(self.l1(self.norm(x))))


class MLPHead(nn.Module):
    def __init__(self, d_in, d_hidden=256, d_out=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in), nn.Linear(d_in, d_hidden), nn.GELU(),
            nn.LayerNorm(d_hidden), nn.Linear(d_hidden, d_out))

    def forward(self, x):
        return self.net(x.float())


class KANMetric(nn.Module):
    """Learnable additive dissimilarity between a query q and a prototype c.

        d(q, c) = ||q - c||^2 + sum_j phi_j(|q_j - c_j| * sqrt(D))

    A single KAN layer with one output is exactly a sum of learned univariate
    functions (the inner sum of the Kolmogorov-Arnold representation), so the
    metric generalises (weighted) squared Euclidean while every phi_j can be
    plotted and inspected.  The KAN term starts at ~0, i.e. training begins
    from a standard Prototypical Network.
    """

    def __init__(self, d, grid_size=5):
        super().__init__()
        self.d = d
        self.kan = KANLinear(d, 1, grid_size=grid_size, grid_range=(0.0, 4.0), scale_noise=0.01)
        with torch.no_grad():
            self.kan.base_weight.zero_()
            self.kan.spline_scaler.mul_(0.1)

    def forward(self, q, c):
        """q: (Q, D), c: (N, D) normalised embeddings -> (Q, N) dissimilarities."""
        with torch.autocast("cuda", enabled=False):
            q, c = q.float(), c.float()
            delta = q[:, None, :] - c[None, :, :]
            euc = delta.pow(2).sum(-1)
            u = (delta.abs() * self.d ** 0.5).clamp(max=3.99).flatten(0, 1)
            return euc + self.kan(u).view(q.shape[0], c.shape[0])


def make_head(kind, d_in, d_hidden=256, d_out=128):
    if kind == "none":
        return nn.Identity(), d_in
    if kind == "mlp":
        return MLPHead(d_in, d_hidden, d_out), d_out
    if kind == "kan":
        return KANHead(d_in, d_hidden, d_out), d_out
    raise ValueError(kind)


# Backbones
class Conv4(nn.Module):
    """The original Prototypical-Network encoder (Snell et al. 2017)."""

    def __init__(self, hid=64):
        super().__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.net = nn.Sequential(block(3, hid), block(hid, hid), block(hid, hid), block(hid, hid))

    def forward(self, x):
        return self.net(x).flatten(1)


BACKBONES = {
    # name: (input resolution, backbone lr)
    "conv4": (84, 1e-3),
    "resnet18": (224, 1e-4),
    "resnet50": (224, 1e-4),
    "densenet121": (224, 1e-4),
    "dinov2_s": (224, 2e-5),
    "dinov2_b": (224, 2e-5),
    # lightweight CNNs used by earlier mango-variety classifiers
    "efficientnet_b0": (224, 1e-4),
    "mobilenet_v2": (224, 1e-4),
}


def make_backbone(name, pretrained=True):
    import torchvision.models as tvm
    if name == "conv4":
        m = Conv4()
        return m, 64 * 5 * 5
    if name == "resnet18":
        m = tvm.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
        d = m.fc.in_features
        m.fc = nn.Identity()
        return m, d
    if name == "resnet50":
        m = tvm.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        d = m.fc.in_features
        m.fc = nn.Identity()
        return m, d
    if name == "densenet121":
        m = tvm.densenet121(weights="IMAGENET1K_V1" if pretrained else None)
        d = m.classifier.in_features
        m.classifier = nn.Identity()
        return m, d
    if name in ("efficientnet_b0", "mobilenet_v2"):
        m = getattr(tvm, name)(weights="IMAGENET1K_V1" if pretrained else None)
        d = m.classifier[-1].in_features
        m.classifier = nn.Identity()
        return m, d
    if name == "dinov2_s":
        import timm
        m = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=pretrained,
                              num_classes=0, img_size=224)
        return m, m.num_features
    if name == "dinov2_b":
        import timm
        m = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=pretrained,
                              num_classes=0, img_size=224)
        m.set_grad_checkpointing(True)  # fits a P x M = 80 batch on a 16 GB T4
        return m, m.num_features
    raise ValueError(name)


class EmbeddingNet(nn.Module):
    def __init__(self, backbone="resnet18", head="none", pretrained=True,
                 d_hidden=256, d_out=128, metric="euclid"):
        super().__init__()
        self.res, self.backbone_lr = BACKBONES[backbone]
        self.backbone, self.d_feat = make_backbone(backbone, pretrained)
        self.head, self.d_emb = make_head(head, self.d_feat, d_hidden, d_out)
        self.metric = KANMetric(self.d_emb) if metric == "kan" else None
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def features(self, x):
        """x: float images in [0,1], already at self.res."""
        return self.backbone((x - self.mean) / self.std).float()

    def forward(self, x, return_feat=False):
        f = self.features(x)
        with torch.autocast("cuda", enabled=False):  # KAN splines need fp32
            z = self.head(f.float()).float()
        return (z, f) if return_feat else z

    def param_groups(self, head_lr):
        heads = list(self.head.parameters())
        if self.metric is not None:
            heads += list(self.metric.parameters())
        return [{"params": self.backbone.parameters(), "lr": self.backbone_lr},
                {"params": heads, "lr": head_lr}]


class FusionNet(nn.Module):
    """Feature-level fusion of several embedding networks, used for evaluation only.

    The backbone features of the members are L2-normalised and concatenated, so each
    member contributes equally; the concatenation is both z and f of the fused model.
    """

    def __init__(self, members):
        super().__init__()
        self.members = nn.ModuleList(members)
        self.res = members[0].res
        assert all(m.res == self.res for m in members)
        self.metric = None

    def forward(self, x, return_feat=False):
        f = torch.cat([F.normalize(m.features(x).float(), dim=1) for m in self.members], 1)
        z = F.normalize(f, dim=1)
        return (z, f) if return_feat else z


def count_params(m):
    return sum(p.numel() for p in m.parameters())

## 2.5 Objectives
Prototypical loss with optional **prototype margin** and learned metric; CE (cosine), ArcFace, SupCon, triplet with four mining policies.

In [ ]:
%%writefile src/losses.py
"""Training objectives compared in the study.

All objectives consume the same class-balanced batch of P classes x M images
(see data.PKSampler), so they see exactly the same images per step.

  ce        softmax cross-entropy with a cosine classifier (Baseline++)
  arcface   additive angular margin (Deng et al. 2019)
  supcon    supervised contrastive (Khosla et al. 2020)
  triplet   triplet margin loss with a selectable mining policy
  proto     Prototypical-Network episodic loss (Snell et al. 2017)
  proto_tri proto + lambda * triplet (aggregated metric objective)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def pdist2(a, b):
    """Squared euclidean distance matrix."""
    return (a.pow(2).sum(1, keepdim=True) - 2 * a @ b.t() + b.pow(2).sum(1).unsqueeze(0)).clamp_min(0)


def triplet_loss(z, y, margin=0.2, mining="batch_hard", cutoff=0.5):
    """z: L2-normalised embeddings (B, d); y: (B,) labels."""
    d = pdist2(z, z).clamp_min(1e-12).sqrt()
    same = y[:, None] == y[None, :]
    eye = torch.eye(len(y), dtype=torch.bool, device=y.device)
    pos_mask, neg_mask = same & ~eye, ~same
    if mining == "batch_hard":
        hp = (d * pos_mask).max(1).values
        hn = d.masked_fill(~neg_mask, float("inf")).min(1).values
        return F.relu(hp - hn + margin).mean()
    # all anchor-positive pairs
    a_idx, p_idx = pos_mask.nonzero(as_tuple=True)
    d_ap = d[a_idx, p_idx]
    d_an = d[a_idx]  # (n_pairs, B)
    nm = neg_mask[a_idx]
    if mining == "random":
        w = nm.float()
    elif mining == "semi_hard":
        # negatives farther than the positive but inside the margin;
        # fall back to the hardest negative when none qualify
        semi = nm & (d_an > d_ap[:, None]) & (d_an < d_ap[:, None] + margin)
        none = ~semi.any(1)
        hard = d_an.masked_fill(~nm, float("inf")).argmin(1)
        semi[none, hard[none]] = True
        w = semi.float()
    elif mining == "distance_weighted":
        # Wu et al. 2017: sample negatives with prob ~ 1/q(d) on the unit sphere
        dim = z.shape[1]
        dc = d_an.clamp_min(cutoff)
        log_w = (2.0 - dim) * dc.log() - ((dim - 3) / 2.0) * (1.0 - 0.25 * dc.pow(2)).clamp_min(1e-8).log()
        log_w = log_w.masked_fill(~nm, float("-inf"))
        w = torch.softmax(log_w - log_w.max(1, keepdim=True).values, 1)
        w = w * (d_an < 1.4).float() + 1e-12 * nm.float()
    else:
        raise ValueError(mining)
    neg = torch.multinomial(w / w.sum(1, keepdim=True), 1).squeeze(1)
    return F.relu(d_ap - d_an.gather(1, neg[:, None]).squeeze(1) + margin).mean()


def proto_logits(zs, ys, zq, n_way, scale, metric=None):
    protos = torch.stack([zs[ys == c].mean(0) for c in range(n_way)])
    d = pdist2(zq, protos) if metric is None else metric(zq, protos)
    return -scale * d


def proto_loss(z, y, n_support, scale, metric=None, margin=0.0):
    """Split every class of the P x M batch into support / query halves.

    margin > 0 gives the prototype margin loss: the true-class distance is
    inflated by `margin` inside the softmax, so a query must be closer to its
    own prototype than to every other prototype by at least that much."""
    if z.device.type == "xla":
        return _proto_loss_static(z, y, n_support, scale, metric, margin)
    classes = y.unique()
    ys, yq, zs, zq = [], [], [], []
    for i, c in enumerate(classes):
        idx = (y == c).nonzero(as_tuple=True)[0]
        zs.append(z[idx[:n_support]]); ys += [i] * n_support
        zq.append(z[idx[n_support:]]); yq += [i] * (len(idx) - n_support)
    zs, zq = torch.cat(zs), torch.cat(zq)
    ys = torch.tensor(ys, device=z.device)
    yq = torch.tensor(yq, device=z.device)
    logits = proto_logits(zs, ys, zq, len(classes), scale, metric)
    acc = (logits.argmax(1) == yq).float().mean()
    if margin > 0:
        logits = logits - scale * margin * F.one_hot(yq, len(classes))
    return F.cross_entropy(logits, yq), acc


def _proto_loss_static(z, y, n_support, scale, metric, margin):
    """proto_loss with shapes fixed by the P x M batch (no data-dependent indexing),
    for XLA devices. The support / query split is computed on the host."""
    yc = y.cpu()
    classes = yc.unique()
    sup, qry, yq = [], [], []
    for i, c in enumerate(classes):
        idx = (yc == c).nonzero(as_tuple=True)[0]
        sup.append(idx[:n_support]); qry.append(idx[n_support:]); yq += [i] * (len(idx) - n_support)
    n_way = len(classes)
    zs = z[torch.cat(sup).to(z.device)]
    zq = z[torch.cat(qry).to(z.device)]
    yq = torch.tensor(yq, device=z.device)
    protos = zs.view(n_way, n_support, -1).mean(1)
    logits = -scale * (pdist2(zq, protos) if metric is None else metric(zq, protos))
    acc = (logits.argmax(1) == yq).float().mean()
    if margin > 0:
        logits = logits - scale * margin * F.one_hot(yq, n_way)
    return F.cross_entropy(logits, yq), acc


def supcon_loss(z, y, t=0.1):
    sim = z @ z.t() / t
    eye = torch.eye(len(y), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, float("-inf"))
    logp = sim - torch.logsumexp(sim, 1, keepdim=True)
    pos = (y[:, None] == y[None, :]) & ~eye
    return -(logp.masked_fill(~pos, 0).sum(1) / pos.sum(1).clamp_min(1)).mean()


class CosineClassifier(nn.Module):
    def __init__(self, d, n_classes, scale=16.0, margin=0.0):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_classes, d) * 0.01)
        self.scale, self.margin = scale, margin

    def forward(self, z, y=None):
        cos = F.normalize(z, dim=1) @ F.normalize(self.w, dim=1).t()
        if self.margin > 0 and y is not None:  # ArcFace: cos(theta + m) on the target
            theta = torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6))
            target = torch.cos(theta + self.margin)
            onehot = F.one_hot(y, cos.shape[1]).bool()
            cos = torch.where(onehot, target, cos)
        return self.scale * cos


class Objective(nn.Module):
    def __init__(self, kind, d_emb, n_classes, n_support=5, tri_margin=0.2,
                 mining="batch_hard", lam=1.0, margin=0.0):
        super().__init__()
        self.kind, self.n_support, self.lam, self.margin = kind, n_support, lam, margin
        self.tri_margin, self.mining = tri_margin, mining
        self.log_scale = nn.Parameter(torch.tensor(2.3))  # learnable temperature, exp(2.3)~10
        if kind in ("ce", "arcface"):
            self.clf = CosineClassifier(d_emb, n_classes, margin=0.3 if kind == "arcface" else 0.0)

    def forward(self, z, y, metric=None):
        """z: raw embeddings, y: global class ids, metric: the model's learned
        distance (None = squared Euclidean). returns (loss, train-acc)."""
        zn = F.normalize(z, dim=1)
        k = self.kind
        if k in ("ce", "arcface"):
            logits = self.clf(z, y)
            acc = (self.clf(z).argmax(1) == y).float().mean()
            return F.cross_entropy(logits, y), acc
        if k == "supcon":
            return supcon_loss(zn, y), torch.tensor(0.0)
        if k == "triplet":
            return triplet_loss(zn, y, self.tri_margin, self.mining), torch.tensor(0.0)
        if k in ("proto", "proto_tri"):
            loss, acc = proto_loss(zn, y, self.n_support, self.log_scale.exp(), metric, self.margin)
            if k == "proto_tri":
                loss = loss + self.lam * triplet_loss(zn, y, self.tri_margin, self.mining)
            return loss, acc
        raise ValueError(k)

## 2.6 Episodic evaluation
5-way K-shot episodes with fixed seeds (identical across models), accuracy / macro-F1 with 95% CI, balanced accuracy, ECE, per-cultivar recall, confusion, open-set AUROC.

In [ ]:
%%writefile src/evaluate.py
"""Episodic few-shot evaluation on frozen embeddings.

For every trained model we embed the test images once and then score
hundreds of N-way K-shot episodes with several inference rules:
  proto_z   nearest prototype on the (normalised) head output z
  proto_f   nearest prototype on the (normalised) backbone feature f
  proto_fc  as proto_f after centring on the base-class mean (SimpleShot CL2N)
  proto_fs  as proto_f after centring on the mean feature of the image's own source
            dataset, computed from the unlabelled test images of that source
            (transductive domain normalisation; no labels are used)
  logreg_z  logistic regression fitted on the support embeddings
The episode seeds depend only on (protocol, fold, K) so every model is scored
on identical episodes - paired comparisons are therefore valid.
"""
import numpy as np
import torch
import torch.nn.functional as F

from data import gpu_augment, sample_episode, eligible_classes


@torch.no_grad()
def embed(model, cache, idx, device, bs=128):
    model.eval()
    zs, fs = [], []
    for s in range(0, len(idx), bs):
        x = gpu_augment(cache.batch(idx[s:s + bs], device), model.res, train=False)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            z, f = model(x, return_feat=True)
        zs.append(z.float().cpu()); fs.append(f.float().cpu())
    return torch.cat(zs), torch.cat(fs)


def macro_f1(y, p, n):
    f1 = []
    for c in range(n):
        tp = ((p == c) & (y == c)).sum()
        fp = ((p == c) & (y != c)).sum()
        fn = ((p != c) & (y == c)).sum()
        f1.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1))


def ci95(a):
    a = np.asarray(a, dtype=float)
    return float(a.mean()), float(1.96 * a.std(ddof=1) / np.sqrt(len(a))) if len(a) > 1 else 0.0


def proto_dist(es, ys, eq, n_way, metric=None):
    es, eq = F.normalize(es, dim=1), F.normalize(eq, dim=1)
    protos = torch.stack([es[ys == c].mean(0) for c in range(n_way)])
    if metric is None:
        return torch.cdist(eq, protos).pow(2)
    with torch.no_grad():
        return metric(eq, protos)


def proto_predict(es, ys, eq, n_way, metric=None):
    """Nearest prototype.  Confidences use a fixed softmax temperature (10) for
    every model, so ECE compares raw distance scales, not calibrated models."""
    d = proto_dist(es, ys, eq, n_way, metric)
    return d.argmin(1), torch.softmax(-10 * d, 1)


def logreg_predict(es, ys, eq, n_way, steps=100, wd=1e-3):
    es, eq = F.normalize(es, dim=1), F.normalize(eq, dim=1)
    W = torch.zeros(es.shape[1], n_way, requires_grad=True)
    b = torch.zeros(n_way, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=steps, line_search_fn="strong_wolfe")
    yt = torch.as_tensor(ys)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(10 * es @ W + b, yt) + wd * W.pow(2).sum()
        loss.backward()
        return loss
    opt.step(closure)
    with torch.no_grad():
        p = torch.softmax(10 * eq @ W + b, 1)
    return p.argmax(1), p


def ece(conf, correct, bins=15):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return float(e)


def run_episodes(Z, Fe, labels, groups, class_names, *, n_way=5, shots=(1, 5, 10), n_query=15,
                 n_episodes=600, seed=0, rules=("proto_z", "proto_f", "logreg_z"),
                 domains=None, domain_pair=None, metric=None, base_mean=None, src_centred=None):
    """Z, Fe: (N, d) tensors aligned with labels/groups (numpy).  Rules are
    '<classifier>_<features>' with features z (head output), f (backbone) or
    fc (backbone centred on base_mean).  Returns a dict."""
    feats = {"z": Z, "f": Fe}
    if base_mean is not None:
        feats["fc"] = F.normalize(Fe, dim=1) - base_mean
    if src_centred is not None:
        feats["fs"] = src_centred
    out = {}
    labels, groups = np.asarray(labels), np.asarray(groups)
    for k in shots:
        if domain_pair is None:
            classes = eligible_classes(labels, groups, k, n_query)
        else:
            a, b = domain_pair
            classes = np.array([c for c in np.unique(labels)
                                if ((labels == c) & (domains == a)).sum() >= k
                                and ((labels == c) & (domains == b)).sum() >= 5])
        nw = min(n_way, len(classes))
        if nw < 2:
            continue
        rng = np.random.default_rng(seed * 1000 + k)
        rec = {r: {"acc": [], "f1": [], "conf": [], "correct": []} for r in rules}
        hits = {r: np.zeros((len(class_names), len(class_names)), dtype=np.int64) for r in rules}
        for _ in range(n_episodes):
            sp, ys, qp, yq, cls = sample_episode(
                rng, labels, groups, classes, nw, k, n_query,
                group_disjoint=domain_pair is None, domains=domains,
                support_domain=None if domain_pair is None else domain_pair[0],
                query_domain=None if domain_pair is None else domain_pair[1])
            ys_t = torch.as_tensor(ys)
            for r in rules:
                E = feats[r.split("_", 1)[1]]
                if r.startswith("proto"):
                    pred, prob = proto_predict(E[sp], ys_t, E[qp], nw,
                                               metric if r == "proto_z" else None)
                else:
                    pred, prob = logreg_predict(E[sp], ys_t, E[qp], nw)
                pred = pred.numpy()
                rec[r]["acc"].append(float((pred == yq).mean()))
                rec[r]["f1"].append(macro_f1(yq, pred, nw))
                rec[r]["conf"] += prob.max(1).values.tolist()
                rec[r]["correct"] += (pred == yq).tolist()
                np.add.at(hits[r], (cls[yq], cls[pred]), 1)
        for r in rules:
            acc, acc_ci = ci95(rec[r]["acc"])
            f1, f1_ci = ci95(rec[r]["f1"])
            conf = hits[r]
            recall = {class_names[i]: float(conf[i, i] / conf[i].sum())
                      for i in range(len(class_names)) if conf[i].sum() > 0}
            out[f"{r}@{k}shot"] = {
                "acc": acc, "acc_ci": acc_ci, "f1": f1, "f1_ci": f1_ci,
                "bal_acc": float(np.mean(list(recall.values()))),
                "ece": ece(rec[r]["conf"], rec[r]["correct"]),
                "n_way": nw, "n_episodes": n_episodes,
                "recall": recall,
                "confusion": conf.tolist() if r == "proto_z" else None,
                # per-episode accuracies -> paired significance tests between
                # models (episodes are identical across models by seeding)
                "episode_acc": [round(a, 4) for a in rec[r]["acc"]] if r.startswith("proto") else None,
            }
    return out


def auroc(pos, neg):
    """P(score_pos > score_neg) with ties counted half (Mann-Whitney)."""
    pos, neg = np.asarray(pos), np.asarray(neg)
    allv = np.concatenate([pos, neg])
    ranks = allv.argsort().argsort().astype(float) + 1
    # average ranks for ties
    _, inv, cnt = np.unique(allv, return_inverse=True, return_counts=True)
    sums = np.zeros(len(cnt)); np.add.at(sums, inv, ranks)
    ranks = (sums / cnt)[inv]
    return float((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


def open_set_eval(Z, labels, groups, n_way=5, shots=(1, 5), n_query=15, n_episodes=300,
                  seed=0, metric=None, name="open"):
    """Open-set episodes: N known cultivars + 1 unknown cultivar whose queries
    must be rejected.  Score = distance to the nearest prototype; we report
    AUROC (unknown vs known queries) and closed-set accuracy on the knowns."""
    out = {}
    labels, groups = np.asarray(labels), np.asarray(groups)
    for k in shots:
        classes = eligible_classes(labels, groups, k, n_query)
        nw = min(n_way, len(classes) - 1)
        if nw < 2:
            continue
        rng = np.random.default_rng(seed * 1000 + k)
        aucs, accs = [], []
        for _ in range(n_episodes):
            sp, ys, qp, yq, _ = sample_episode(rng, labels, groups, classes, nw + 1, k, n_query)
            known_s = ys < nw
            d = proto_dist(Z[sp[known_s]], torch.as_tensor(ys[known_s]), Z[qp], nw, metric)
            score = d.min(1).values.numpy()
            unk = yq == nw
            aucs.append(auroc(score[unk], score[~unk]))
            accs.append(float((d.argmin(1).numpy()[~unk] == yq[~unk]).mean()))
        a, ac = ci95(aucs)
        b, bc = ci95(accs)
        out[f"{name}@{k}shot"] = {"auroc": a, "auroc_ci": ac, "closed_acc": b, "closed_acc_ci": bc,
                                "n_way": nw, "n_episodes": n_episodes,
                                "episode_auroc": [round(x, 4) for x in aucs]}
    return out

## 2.7 Training one configuration

In [ ]:
%%writefile src/train.py
"""Train one embedding model on base cultivars and evaluate it on test episodes."""
import copy
import math
import time
import zlib

import numpy as np
import torch

from data import PKSampler, gpu_augment, protocol_split
from evaluate import embed, open_set_eval, run_episodes
from losses import Objective
from models import EmbeddingNet, count_params

DEFAULTS = dict(
    protocol="unified", fold=0, backbone="resnet18", head="none", objective="proto",
    steps=1200, P=8, M=10, n_support=5, head_lr=1e-3, wd=1e-4, warmup=0.05,
    mining="batch_hard", tri_margin=0.2, lam=1.0, pretrained=True, seed=0,
    variant="crop", n_episodes=600, shots=(1, 5, 10), n_way=5,
    metric="euclid", margin=0.0,
)


def set_seed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def train_model(cache, cfg, device, train_idx, log_every=200):
    set_seed(cfg["seed"])
    if device.type == "xla":
        import torch_xla.core.xla_model as xm
        xm.set_rng_state(cfg["seed"])
    meta = cache.meta.set_index("idx").loc[train_idx]
    names = sorted(meta.cultivar.unique())
    y_all = meta.cultivar.map({n: i for i, n in enumerate(names)}).to_numpy()
    model = EmbeddingNet(cfg["backbone"], cfg["head"], cfg["pretrained"],
                         metric=cfg["metric"]).to(device)
    obj = Objective(cfg["objective"], model.d_emb, len(names), cfg["n_support"],
                    cfg["tri_margin"], cfg["mining"], cfg["lam"], cfg["margin"]).to(device)
    groups = model.param_groups(cfg["head_lr"])
    if not cfg["pretrained"]:
        groups[0]["lr"] = cfg["head_lr"]
    groups.append({"params": obj.parameters(), "lr": cfg["head_lr"]})
    wd = 0.05 if cfg["backbone"].startswith("dino") else cfg["wd"]
    opt = torch.optim.AdamW(groups, weight_decay=wd)
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    sampler = PKSampler(y_all, cfg["P"], cfg["M"], seed=cfg["seed"])
    steps, warm = cfg["steps"], max(1, int(cfg["warmup"] * cfg["steps"]))
    hist, t0 = [], time.time()
    model.train()
    for step in range(steps):
        lr_f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / (steps - warm)))
        for g, b in zip(opt.param_groups, base_lrs):
            g["lr"] = b * lr_f
        pos = sampler.sample()
        x = gpu_augment(cache.batch(train_idx[pos], device), model.res, train=True)
        y = torch.as_tensor(y_all[pos], device=device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            z = model(x)
        loss, acc = obj(z.float(), y, model.metric)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()
        if device.type == "xla":
            import torch_xla.core.xla_model as xm
            xm.mark_step()
        if step % log_every == 0 or step == steps - 1:
            hist.append((step, float(loss), float(acc), time.time() - t0))
            print(f"  step {step} loss {float(loss):.4f} acc {float(acc):.3f} {time.time() - t0:.0f}s", flush=True)
    return model, {"history": hist, "train_time_s": time.time() - t0, "n_base": len(names)}


def base_feature_mean(model, cache, train_idx, device, n=2000, seed=0):
    """Mean of L2-normalised backbone features over (a sample of) base images."""
    sub = np.random.default_rng(seed).choice(train_idx, min(n, len(train_idx)), replace=False)
    _, Fb = embed(model, cache, np.sort(sub), device)
    return torch.nn.functional.normalize(Fb, dim=1).mean(0)


def evaluate_model(model, cache, cfg, device, test_idx, info, train_idx, extra_eval=True):
    meta = cache.meta.set_index("idx").loc[test_idx]
    Z, Fe = embed(model, cache, test_idx, device)
    base_mean = base_feature_mean(model, cache, train_idx, device)
    names = sorted(meta.cultivar.unique())
    labels = meta.cultivar.map({n: i for i, n in enumerate(names)}).to_numpy()
    groups = meta.group.to_numpy()
    dom = meta.dataset.to_numpy()
    # transductive domain normalisation: centre each image on the mean feature of
    # its source dataset over the (unlabelled) test images of that source
    Fn = torch.nn.functional.normalize(Fe, dim=1)
    Fs = Fn.clone()
    for s_ in np.unique(dom):
        Fs[dom == s_] -= Fn[dom == s_].mean(0)
    seed_base = zlib.crc32(f'{cfg["protocol"]}|{cfg["fold"]}'.encode()) % 9973
    metric = copy.deepcopy(model.metric).cpu().eval() if model.metric is not None else None
    res = {"all": run_episodes(Z, Fe, labels, groups, names, n_way=cfg["n_way"], shots=cfg["shots"],
                               n_episodes=cfg["n_episodes"], seed=seed_base, metric=metric,
                               rules=("proto_z", "proto_f", "proto_fc", "proto_fs", "logreg_z"),
                               base_mean=base_mean, src_centred=Fs)}
    if extra_eval:
        res["open_set"] = open_set_eval(Z, labels, groups, n_way=cfg["n_way"], shots=(1, 5),
                                        n_episodes=300, seed=seed_base + 3, metric=metric)
        for nm, E in (("open_f", Fn), ("open_fs", Fs)):
            res["open_set"].update(open_set_eval(E, labels, groups, n_way=cfg["n_way"], shots=(1, 5),
                                                 n_episodes=300, seed=seed_base + 3, name=nm))
    if cfg["protocol"] == "lodo" and extra_eval:
        for part in ("seen", "novel"):
            keep = np.isin(meta.cultivar.to_numpy(), info[part])
            if len(np.unique(labels[keep])) >= 2:
                res[part] = run_episodes(Z[keep], Fe[keep], labels[keep], groups[keep], names,
                                         n_way=cfg["n_way"], shots=cfg["shots"],
                                         n_episodes=cfg["n_episodes"], seed=seed_base + 1,
                                         rules=("proto_z", "proto_f", "proto_fc", "proto_fs"), metric=metric,
                                         base_mean=base_mean, src_centred=Fs[keep])
    if cfg["protocol"] == "unified" and extra_eval:
        # cross-domain episodes: support from dataset a, query from dataset b
        for a in np.unique(dom):
            for b in np.unique(dom):
                if a == b:
                    continue
                r = run_episodes(Z, Fe, labels, groups, names, n_way=cfg["n_way"], shots=(1, 5),
                                 n_episodes=300, seed=seed_base + 7, rules=("proto_z", "proto_f", "proto_fs"),
                                 domains=dom, domain_pair=(a, b), metric=metric, src_centred=Fs)
                if r:
                    res[f"x:{a}->{b}"] = r
    return res, (Z, Fe, labels, groups, names)


def run_config(cache, cfg, device, save_ckpt=None):
    cfg = {**DEFAULTS, **cfg}
    tr, te, info = protocol_split(cache.meta, cfg["protocol"], cfg["fold"])
    t0 = time.time()
    model, tinfo = train_model(cache, cfg, device, tr)
    res, _ = evaluate_model(model, cache, cfg, device, te, info, tr)
    if save_ckpt:
        torch.save({k: (v.half() if v.is_floating_point() else v).cpu()
                    for k, v in model.state_dict().items()},
                   save_ckpt)
    return {
        "cfg": {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()},
        "split": info, "n_train": len(tr), "n_test": len(te),
        "params_backbone": count_params(model.backbone), "params_head": count_params(model.head),
        "train": tinfo, "results": res, "wall_s": time.time() - t0,
    }


def run_zero_shot(cache, cfg, device):
    """ImageNet-pretrained backbone, no training on mango data."""
    cfg = {**DEFAULTS, **cfg, "head": "none", "objective": "imagenet", "steps": 0}
    tr, te, info = protocol_split(cache.meta, cfg["protocol"], cfg["fold"])
    model = EmbeddingNet(cfg["backbone"], "none", True).to(device)
    res, _ = evaluate_model(model, cache, cfg, device, te, info, tr)
    return {"cfg": {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()},
            "split": info, "n_train": 0, "n_test": len(te),
            "params_backbone": count_params(model.backbone), "params_head": 0,
            "train": {}, "results": res, "wall_s": 0}

## 2.8 Normal fine-tuning baselines
Per-episode full fine-tuning on the support set, and the closed-set study (K images per cultivar) incl. random-split leakage inflation.

In [ ]:
%%writefile src/finetune.py
"""'Normal' fine-tuning baselines and the closed-set limited-data study.

1. episode_finetune  - for novel-cultivar episodes: copy an ImageNet (or
   meta-trained) network, attach a fresh classifier and fine-tune the whole
   thing on the K-shot support set, then predict the queries.
2. closed_set_study  - all cultivars of one dataset are known; we train with
   K images per cultivar and test on held-out capture groups.  Compares
   ordinary fine-tuning against prototype classification with an embedding
   meta-trained on the *other* datasets (no labels of the target dataset are
   used except the K shots).
"""
import copy
import time

import numpy as np
import torch
import torch.nn.functional as F

from data import gpu_augment, sample_episode, eligible_classes
from evaluate import embed, macro_f1, ci95, proto_predict
from losses import CosineClassifier
from models import EmbeddingNet


def finetune_classifier(model, cache, train_idx, y, n_cls, device, steps, bs=32,
                        head_lr=1e-3, freeze_backbone=False):
    model = copy.deepcopy(model).to(device).train()
    clf = CosineClassifier(model.d_emb, n_cls).to(device)
    if freeze_backbone:
        for p in model.backbone.parameters():
            p.requires_grad_(False)
    groups = [{"params": list(model.head.parameters()) + list(clf.parameters()), "lr": head_lr}]
    if not freeze_backbone:
        groups.append({"params": model.backbone.parameters(), "lr": model.backbone_lr})
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    y = np.asarray(y)
    rng = np.random.default_rng(0)
    for _ in range(steps):
        pos = rng.choice(len(train_idx), bs, replace=len(train_idx) < bs)
        x = gpu_augment(cache.batch(train_idx[pos], device), model.res, train=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            z = model(x)
        loss = F.cross_entropy(clf(z.float()), torch.as_tensor(y[pos], device=device))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
    model.eval()

    def predict(idx):
        Z, _ = embed(model, cache, idx, device)
        with torch.no_grad():
            return clf(Z.to(device)).softmax(1).cpu()
    return predict


def episode_finetune(cache, test_idx, base_model, device, shots=(1, 5), n_episodes=100, n_way=5,
                     n_query=15, steps=100, seed=0):
    """Full fine-tuning on each support set (the 'normal fine-tuning' arm)."""
    meta = cache.meta.set_index("idx").loc[test_idx]
    names = sorted(meta.cultivar.unique())
    labels = meta.cultivar.map({n: i for i, n in enumerate(names)}).to_numpy()
    groups = meta.group.to_numpy()
    out = {}
    for k in shots:
        classes = eligible_classes(labels, groups, k, n_query)
        rng = np.random.default_rng(seed * 1000 + k)
        accs, f1s, t0 = [], [], time.time()
        for e in range(n_episodes):
            sp, ys, qp, yq, _ = sample_episode(rng, labels, groups, classes, n_way, k, n_query)
            pred_fn = finetune_classifier(base_model, cache, test_idx[sp], ys, n_way, device, steps)
            pred = pred_fn(test_idx[qp]).argmax(1).numpy()
            accs.append(float((pred == yq).mean()))
            f1s.append(macro_f1(yq, pred, n_way))
        a, ac = ci95(accs)
        f, fc = ci95(f1s)
        out[f"finetune@{k}shot"] = {"acc": a, "acc_ci": ac, "f1": f, "f1_ci": fc,
                                    "n_episodes": n_episodes, "n_way": n_way,
                                    "sec_per_episode": (time.time() - t0) / n_episodes}
    return out


def group_holdout(meta, frac=0.3, seed=0, naive=False):
    """Per-cultivar split; whole capture groups go to test until ~frac."""
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for c, m in meta.groupby("cultivar"):
        if naive:
            idx = rng.permutation(m.idx.to_numpy())
            n = int(round(frac * len(idx)))
            te += idx[:n].tolist(); tr += idx[n:].tolist()
            continue
        g = m.groupby("group").idx.apply(list)
        order = rng.permutation(len(g))
        n_te, target = 0, frac * len(m)
        for i in order:
            members = g.iloc[i]
            if n_te < target:
                te += members; n_te += len(members)
            else:
                tr += members
    return np.array(tr), np.array(te)


def pick_k(meta_tr, k, rng):
    """K training images per cultivar, spread over as many groups as possible."""
    out = []
    for c, m in meta_tr.groupby("cultivar"):
        if k == "all":
            out += m.idx.tolist(); continue
        firsts = m.sample(frac=1, random_state=int(rng.integers(1 << 30))).drop_duplicates("group")
        pick = firsts.idx.tolist()[:k]
        if len(pick) < k:
            rest = m[~m.idx.isin(pick)].idx.to_numpy()
            pick += rng.choice(rest, min(k - len(pick), len(rest)), replace=False).tolist()
        out += pick
    return np.array(out)


def score(probs, y, n):
    pred = probs.argmax(1).numpy()
    rec = [float((pred[y == c] == c).mean()) for c in range(n) if (y == c).any()]
    return {"acc": float((pred == y).mean()), "f1": macro_f1(y, pred, n), "bal_acc": float(np.mean(rec))}


def closed_set_study(cache, dataset, device, backbone, heads, ks, seeds, meta_ckpt=None,
                     naive=False, ft_steps=None):
    meta = cache.meta[(cache.meta.dataset == dataset) & (cache.meta["drop"] == 0)]
    names = sorted(meta.cultivar.unique())
    cmap = {n: i for i, n in enumerate(names)}
    tr_all, te = group_holdout(meta, seed=0, naive=naive)
    mi = cache.meta.set_index("idx")
    y_te = mi.loc[te].cultivar.map(cmap).to_numpy()
    rows = []
    for k in ks:
        for seed in (seeds if k != "all" else seeds[:1]):
            rng = np.random.default_rng(seed)
            tr = pick_k(mi.loc[tr_all].reset_index(), k, rng)
            y_tr = mi.loc[tr].cultivar.map(cmap).to_numpy()
            n_img = len(tr)
            steps = ft_steps or int(np.clip(20 * n_img / 32, 150, 1500))
            # (a) normal fine-tuning from ImageNet, one run per head
            for head in heads:
                torch.manual_seed(seed)
                m0 = EmbeddingNet(backbone, head, True)
                t0 = time.time()
                pf = finetune_classifier(m0, cache, tr, y_tr, len(names), device, steps)
                rows.append({"method": "finetune_imagenet", "head": head, "k": k, "seed": seed,
                             **score(pf(te), y_te, len(names)), "sec": time.time() - t0})
            # (b) prototypes on frozen ImageNet features (no training)
            m0 = EmbeddingNet(backbone, "none", True).to(device)
            rows.append({"method": "proto_imagenet", "head": "none", "k": k, "seed": seed,
                         **proto_rows(m0, cache, tr, y_tr, te, y_te, len(names), device)})
            # (c) prototypes / fine-tuning from the meta-trained embedding
            if meta_ckpt:
                for src_method, path in meta_ckpt.items():
                    sd = torch.load(path, map_location="cpu")
                    head = "kan" if any(k_.startswith("head.l1.") for k_ in sd) else "mlp"
                    metric = "kan" if any(k_.startswith("metric.") for k_ in sd) else "euclid"
                    mm = EmbeddingNet(backbone, head, False, metric=metric)
                    mm.load_state_dict({k_: v.float() if v.is_floating_point() else v for k_, v in sd.items()})
                    mm = mm.to(device)
                    rows.append({"method": f"proto_meta[{src_method}]", "head": head, "k": k, "seed": seed,
                                 **proto_rows(mm, cache, tr, y_tr, te, y_te, len(names), device)})
                    t0 = time.time()
                    pf = finetune_classifier(mm, cache, tr, y_tr, len(names), device, steps)
                    rows.append({"method": f"finetune_meta[{src_method}]", "head": head, "k": k, "seed": seed,
                                 **score(pf(te), y_te, len(names)), "sec": time.time() - t0})
            print(dataset, k, seed, rows[-1], flush=True)
    for r in rows:
        r.update({"dataset": dataset, "backbone": backbone, "naive_split": naive,
                  "n_train_pool": len(tr_all), "n_test": len(te)})
    return rows


@torch.no_grad()
def proto_rows(model, cache, tr, y_tr, te, y_te, n, device):
    Zs, _ = embed(model, cache, tr, device)
    Zq, _ = embed(model, cache, te, device)
    metric = copy.deepcopy(model.metric).cpu() if model.metric is not None else None
    _, p = proto_predict(Zs, torch.as_tensor(y_tr), Zq, n, metric)
    return score(p, y_te, n)

## 2.9 Cost: parameters, GMACs, latency

In [ ]:
%%writefile src/efficiency.py
"""Deployment cost of every backbone x head (x metric) combination.

Reports parameters, forward GFLOPs at the model's input resolution, GPU
latency (batch 1 and batch 32) and CPU latency (batch 1, 4 threads), plus
the extra cost of scoring one 5-way query with the KAN metric.
"""
import time

import torch
from torch.utils.flop_counter import FlopCounterMode

from models import EmbeddingNet, count_params

COMBOS = [(bb, h, mt) for bb in ["conv4", "resnet18", "resnet50", "densenet121", "dinov2_s", "dinov2_b"]
          for h, mt in [("none", "euclid"), ("mlp", "euclid"), ("kan", "euclid"), ("mlp", "kan")]]


def _latency(fn, n_warm=5, n_iter=30, sync=None):
    for _ in range(n_warm):
        fn()
    if sync:
        sync()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        fn()
    if sync:
        sync()
    return (time.perf_counter() - t0) / n_iter * 1000


@torch.no_grad()
def measure(bb, head, metric, device):
    m = EmbeddingNet(bb, head, pretrained=False, metric=metric).eval()
    r = m.res
    row = {"backbone": bb, "head": head, "metric": metric,
           "params_backbone_M": count_params(m.backbone) / 1e6,
           "params_head_M": count_params(m.head) / 1e6,
           "params_metric_M": count_params(m.metric) / 1e6 if m.metric is not None else 0.0}
    x1 = torch.rand(1, 3, r, r)
    fc = FlopCounterMode(display=False)
    with fc:
        m(x1)
    row["gflops"] = fc.get_total_flops() / 1e9
    row["gmacs"] = fc.get_total_flops() / 2e9
    torch.set_num_threads(4)
    row["cpu_ms_b1"] = _latency(lambda: m(x1), n_iter=10)
    if device.type == "cuda":
        mg = m.to(device)
        xg1, xg32 = x1.to(device), torch.rand(32, 3, r, r, device=device)
        sync = torch.cuda.synchronize
        with torch.autocast("cuda", dtype=torch.float16):
            row["gpu_ms_b1"] = _latency(lambda: mg(xg1), sync=sync)
            row["gpu_ms_b32"] = _latency(lambda: mg(xg32), sync=sync)
        torch.cuda.reset_peak_memory_stats()
        with torch.autocast("cuda", dtype=torch.float16):
            mg(xg32)
        row["gpu_peak_mem_mb_b32"] = torch.cuda.max_memory_allocated() / 2 ** 20
        if mg.metric is not None:
            q, c = torch.randn(75, mg.d_emb, device=device), torch.randn(5, mg.d_emb, device=device)
            row["metric_ms_75x5"] = _latency(lambda: mg.metric(q, c), sync=sync)
        m = mg.cpu()
    return row


def run_all(device):
    rows = []
    for bb, h, mt in COMBOS:
        try:
            rows.append(measure(bb, h, mt, device))
            print(rows[-1], flush=True)
        except Exception as e:  # keep going; report the failure
            rows.append({"backbone": bb, "head": h, "metric": mt, "error": repr(e)})
    return rows

## 2.10 Experiment suites and the resumable driver (multi-GPU or TPU VM)

In [ ]:
%%writefile src/run.py
"""Experiment driver used inside Kaggle kernels.

    import run; run.main("suite_name")

* finds the image cache (output of the prep kernel) under /kaggle/input
* expands the suite into a list of configs
* shards configs over all visible GPUs (one worker process per GPU)
* appends one JSON line per finished config to /kaggle/working/results_<gpu>.jsonl
* skips configs whose key already appears in any results*.jsonl under
  /kaggle/input, so a suite can be resumed by attaching the previous run.
"""
import glob
import hashlib
import json
import os
import subprocess
import sys
import time
import traceback

# Kaggle paths by default; override to run the same driver locally.
INPUT = os.environ.get("MANGOFS_INPUT", "/kaggle/input")
WORK = os.environ.get("MANGOFS_WORK", "/kaggle/working")


def find_root(variant="crop"):
    """Folder of the image cache that holds <variant>.npy next to meta.csv."""
    hits = [os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/meta.csv", recursive=True)]
    hits = [h for h in hits if os.path.exists(os.path.join(h, f"{variant}.npy"))] or hits
    if not hits:
        sys.exit(f"image cache (meta.csv) not found in {INPUT}")
    return hits[0]


def find_groups():
    hits = glob.glob(f"{INPUT}/**/groups.csv", recursive=True) + glob.glob(f"{WORK}/groups.csv")
    return hits[0] if hits else None


def done_keys():
    keys = set()
    for p in glob.glob(f"{INPUT}/**/results*.jsonl", recursive=True) + glob.glob(f"{WORK}/results*.jsonl"):
        with open(p) as f:
            for line in f:
                try:
                    keys.add(json.loads(line)["key"])
                except Exception:
                    pass
    return keys


BACKBONES = ["conv4", "resnet18", "resnet50", "densenet121", "dinov2_s"]

# Named methods.  Everything that is not the proposed method is a baseline
# or an ablation that isolates one ingredient of it.
METHODS = {
    "ce":            dict(objective="ce", head="mlp"),               # fine-tuned classifier (Baseline++)
    "proto":         dict(objective="proto", head="none"),           # vanilla ProtoNet
    "proto_mlp":     dict(objective="proto", head="mlp"),            # ProtoNet + MLP projection
    "proto_kanhead": dict(objective="proto", head="kan"),            # ProtoNet + KAN projection
    "proto_margin":  dict(objective="proto", head="mlp", margin=0.1),  # + margin only
    "kan_metric":    dict(objective="proto", head="mlp", metric="kan"),  # + KAN metric only
    "ours":          dict(objective="proto", head="mlp", metric="kan", margin=0.1),
    # extra baselines: other metric-learning objectives
    "arcface":       dict(objective="arcface", head="mlp"),
    "supcon":        dict(objective="supcon", head="mlp"),
    "triplet":       dict(objective="triplet", head="mlp"),
    "proto_tri":     dict(objective="proto_tri", head="mlp"),
    "ours_kanhead":  dict(objective="proto", head="kan", metric="kan", margin=0.1),
}
MAIN = ["ce", "proto", "proto_mlp", "proto_kanhead", "proto_margin", "kan_metric", "ours"]
# backbones whose features are concatenated in the fusion evaluation (all pretrained ones)
FUSION = ["resnet18", "resnet50", "densenet121", "dinov2_s", "dinov2_b"]
LIGHT = ["efficientnet_b0", "mobilenet_v2"]
EXTRA = ["arcface", "supcon", "triplet", "proto_tri", "ours_kanhead"]


def steps_for(bb):
    return 3000 if bb == "conv4" else 1200


def m(method, **kw):
    return dict(METHODS[method], method=method, **kw)


def suite(name):
    U = dict(protocol="unified")
    if name == "groups":
        return []  # main() builds groups.csv before dispatching
    if name == "smoke":
        return [m(x, **U, fold=0, backbone="resnet18", steps=60, n_episodes=50)
                for x in ["proto_mlp", "ours", "proto_kanhead"]] + \
               [dict(kind="zeroshot", **U, fold=0, backbone="resnet18", n_episodes=50)]
    if name == "unified_main":
        cfgs = []
        for fold in range(3):
            for bb in BACKBONES:
                if bb != "conv4":
                    cfgs.append(dict(kind="zeroshot", **U, fold=fold, backbone=bb))
                for x in MAIN:
                    cfgs.append(m(x, **U, fold=fold, backbone=bb, steps=steps_for(bb)))
        return cfgs
    if name == "unified_extra":
        cfgs = [m(x, **U, fold=f, backbone=bb) for f in range(3)
                for bb in ["resnet18", "densenet121"] for x in EXTRA]
        # margin sensitivity
        cfgs += [dict(m("ours", **U, fold=f, backbone="resnet18"), margin=mg)
                 for f in range(3) for mg in (0.05, 0.2, 0.4)]
        # triplet mining policies (RQ3)
        cfgs += [dict(m("triplet", **U, fold=f, backbone="resnet18"), mining=mn)
                 for f in range(3) for mn in ("random", "semi_hard", "distance_weighted")]
        return cfgs
    if name == "ablations":
        cfgs = []
        for f in range(3):
            for bb in ["resnet18", "densenet121"]:
                for x in ["proto_mlp", "ours"]:
                    cfgs.append(m(x, **U, fold=f, backbone=bb, variant="full"))  # background kept
                for x in ["proto", "proto_mlp", "ours"]:
                    for s_ in (1, 2):
                        cfgs.append(m(x, **U, fold=f, backbone=bb, seed=s_))   # seed variance
        return cfgs
    # follow-up experiments after comparing with other mango-variety pipelines:
    # background removed (seg.npy, prep_seg.py), lightweight CNNs, feature fusion
    if name == "seg_zero":  # no training; runs on CPU
        cfgs = [dict(kind="zeroshot", **U, fold=f, backbone=bb, variant="seg", eval=2)
                for f in range(3) for bb in FUSION + LIGHT]
        cfgs += [dict(kind="zeroshot", **U, fold=f, backbone=bb, eval=2)
                 for f in range(3) for bb in LIGHT]
        return cfgs
    if name == "fusion":  # no training; runs on CPU
        return [dict(kind="fusion", **U, fold=f, members=mb)
                for mb in ("imagenet", "ce") for f in range(3)]
    if name == "seg_train":  # GPU
        return [m(x, **U, fold=f, backbone=bb, variant="seg", eval=2) for f in range(3)
                for bb in ["resnet18", "densenet121"] for x in ["ce", "proto", "proto_mlp"]]
    if name == "light_train":  # GPU
        return [m(x, **U, fold=f, backbone=bb, eval=2) for f in range(3)
                for bb in LIGHT for x in ["ce", "proto", "proto_mlp"]]
    if name == "lodo_main":
        cfgs = []
        for fold in range(3):
            for bb in ["resnet18", "densenet121", "resnet50", "dinov2_s"]:
                cfgs.append(dict(kind="zeroshot", protocol="lodo", fold=fold, backbone=bb))
                for x in ["ce", "proto_mlp", "ours"]:
                    cfgs.append(m(x, protocol="lodo", fold=fold, backbone=bb))
        return cfgs
    if name == "improve":
        # (1) re-evaluate the saved models with the backbone-feature and source-centred
        #     rules (identical episodes; no retraining)
        cfgs = [dict(kind="reeval", ckpt=ck) for ck in input_ckpts(
            lambda c: c["backbone"] in BACKBONES and c.get("variant", "crop") == "crop"
            and c.get("seed", 0) == 0 and c["method"] in ("ce", "proto", "proto_mlp", "ours"))]
        # (2) untrained features with the new rules, plus the larger DINOv2 ViT-B/14
        for p_ in ("unified", "lodo"):
            for fold in range(3):
                for bb in ["resnet18", "resnet50", "densenet121", "dinov2_s", "dinov2_b"]:
                    cfgs.append(dict(kind="zeroshot", protocol=p_, fold=fold, backbone=bb, eval=2))
        for fold in range(3):
            for x in ["proto", "proto_mlp", "ce"]:
                cfgs.append(m(x, **U, fold=fold, backbone="dinov2_b", eval=2))
        return cfgs
    if name == "episode_ft":
        return [dict(kind="episode_ft", protocol="unified", fold=f, backbone=bb)
                for f in range(3) for bb in ["resnet18", "densenet121"]]
    if name == "efficiency":
        return [dict(kind="efficiency")]
    if name == "closed_set":
        return [dict(kind="closed", dataset=ds, backbone=bb, naive=nv)
                for ds in ["MangoImageBD", "MangoClassify12", "Mangifera2012"]
                for bb in ["resnet18", "densenet121"] for nv in (False, True)]
    raise ValueError(name)


def input_records():
    for p in glob.glob(f"{INPUT}/**/results*.jsonl", recursive=True):
        with open(p) as f:
            for line in f:
                try:
                    yield json.loads(line)
                except Exception:
                    pass


def input_ckpts(keep):
    """Checkpoint names of attached trained models whose config satisfies keep(cfg)."""
    out = []
    for r in input_records():
        if r.get("kind") == "train" and r.get("ckpt") and keep({**r["cfg"], "method": r["method"]}):
            out.append(r["ckpt"])
    return sorted(set(out))


def reevaluate(ca, ckpt, dev):
    import torch
    import train as T
    from data import protocol_split
    from models import EmbeddingNet
    rec = next(r for r in input_records() if r.get("ckpt") == ckpt)
    cfg = {**rec["cfg"], "eval": 2}
    path = glob.glob(f"{INPUT}/**/ckpt/{ckpt}", recursive=True)[0]
    model = EmbeddingNet(cfg["backbone"], cfg["head"], pretrained=False, metric=cfg["metric"])
    model.load_state_dict({k: v.float() for k, v in torch.load(path, map_location="cpu").items()})
    model.to(dev)
    tr, te, info = protocol_split(ca.meta, cfg["protocol"], cfg["fold"])
    res, _ = T.evaluate_model(model, ca, cfg, dev, te, info, tr)
    return {"cfg": cfg, "split": info, "n_train": len(tr), "n_test": len(te),
            "params_backbone": rec["params_backbone"], "params_head": rec["params_head"],
            "train": {}, "results": res, "reeval_of": rec["key"], "method": rec["method"]}


def load_trained(protocol, fold, backbone, method):
    """Saved seed-0 crop model of the attached runs (fp16 state dict -> fp32)."""
    import torch
    from models import EmbeddingNet
    rec = next(r for r in input_records() if r.get("kind") == "train" and r.get("ckpt")
               and r["method"] == method and r["cfg"]["protocol"] == protocol
               and r["cfg"]["fold"] == fold and r["cfg"]["backbone"] == backbone
               and r["cfg"].get("seed", 0) == 0 and r["cfg"].get("variant", "crop") == "crop")
    cfg = rec["cfg"]
    path = glob.glob(f"{INPUT}/**/ckpt/{rec['ckpt']}", recursive=True)[0]
    model = EmbeddingNet(cfg["backbone"], cfg["head"], pretrained=False, metric=cfg["metric"])
    model.load_state_dict({k: v.float() for k, v in torch.load(path, map_location="cpu").items()})
    return model, rec["ckpt"]


def fusion_eval(ca, c, dev):
    import train as T
    from data import protocol_split
    from models import EmbeddingNet, FusionNet, count_params
    if c["members"] == "imagenet":
        members, src = [EmbeddingNet(b, "none", True) for b in FUSION], []
    else:
        pairs = [load_trained(c["protocol"], c["fold"], b, c["members"]) for b in FUSION]
        members, src = [p[0] for p in pairs], [p[1] for p in pairs]
    model = FusionNet(members).to(dev)
    cfg = {**T.DEFAULTS, **c, "backbone": "fusion", "head": "none", "eval": 2}
    tr, te, info = protocol_split(ca.meta, cfg["protocol"], cfg["fold"])
    res, _ = T.evaluate_model(model, ca, cfg, dev, te, info, tr)
    return {"cfg": cfg, "split": info, "n_train": 0, "n_test": len(te),
            "params_backbone": count_params(model), "params_head": 0, "train": {},
            "results": res, "members": FUSION, "ckpts": src,
            "method": f"fusion_{c['members']}"}


def on_tpu():
    return bool(os.environ.get("TPU_ACCELERATOR_TYPE") or os.environ.get("PJRT_DEVICE") == "TPU")


def pick_device():
    """CUDA if present, else the TPU (torch_xla) of a TPU VM, else CPU."""
    import torch
    if torch.cuda.is_available():
        return torch.device("cuda")
    if on_tpu():
        import torch_xla.core.xla_model as xm
        return xm.xla_device()
    return torch.device("cpu")


def ckey(c):
    return json.dumps(c, sort_keys=True)


def worker(name, shard, n_shards):
    import torch
    from data import Cache
    import train as T
    dev = pick_device()
    cache_by_variant = {}

    def cache(variant="crop"):
        if variant not in cache_by_variant:
            cache_by_variant[variant] = Cache(find_root(variant), variant, find_groups())
        return cache_by_variant[variant]
    done = done_keys()
    cfgs = suite(name)
    mine = [c for i, c in enumerate(cfgs) if i % n_shards == shard]
    out = open(f"{WORK}/results_{name}_{shard}.jsonl", "a")
    os.makedirs(f"{WORK}/ckpt", exist_ok=True)
    for i, c in enumerate(mine):
        c = dict(c)
        kind = c.pop("kind", "train")
        method = c.pop("method", None)
        key = ckey({"kind": kind, "method": method, **c})
        if key in done:
            continue
        print(f"[gpu{shard}] {i + 1}/{len(mine)} {key}", flush=True)
        t0 = time.time()
        try:
            ca = cache(c.get("variant", "crop"))
            if kind == "train":
                # every trained model is kept (fp16) for re-evaluation and analysis
                tag = hashlib.md5(key.encode()).hexdigest()[:8]
                ck = f"{WORK}/ckpt/{c['protocol']}{c['fold']}_{c['backbone']}_{method}_{tag}.pt"
                r = T.run_config(ca, c, dev, save_ckpt=ck)
                r["ckpt"] = os.path.basename(ck)
            elif kind == "reeval":
                r = reevaluate(ca, c["ckpt"], dev)
            elif kind == "fusion":
                r = fusion_eval(ca, c, dev)
            elif kind == "zeroshot":
                r = T.run_zero_shot(ca, c, dev)
            elif kind == "episode_ft":
                from finetune import episode_finetune
                from data import protocol_split
                from models import EmbeddingNet
                tr, te, info = protocol_split(ca.meta, c["protocol"], c["fold"])
                r = {"results": episode_finetune(ca, te, EmbeddingNet(c["backbone"], "none", True), dev)}
            elif kind == "closed":
                from finetune import closed_set_study
                from data import DATASETS
                held = DATASETS.index(c["dataset"])
                metas = {x: p for x in ["proto_mlp", "ours"] for p in glob.glob(
                    f"{INPUT}/**/ckpt/lodo{held}_{c['backbone']}_{x}_*.pt", recursive=True)}
                r = {"rows": closed_set_study(ca, c["dataset"], dev, c["backbone"], ["mlp", "kan"],
                                              ["all"] if c["naive"] else [1, 5, 10, 20, "all"],
                                              [0, 1, 2], meta_ckpt=None if c["naive"] else metas,
                                              naive=c["naive"])}
            elif kind == "efficiency":
                from efficiency import run_all
                r = {"rows": run_all(dev)}
            r["key"] = key
            r["kind"] = kind
            r["method"] = method or r.get("method")
            r["cfg_in"] = c
            r["elapsed_s"] = time.time() - t0
            out.write(json.dumps(r) + "\n")
            out.flush()
        except Exception:
            traceback.print_exc()
            out_err = open(f"{WORK}/errors_{shard}.txt", "a")
            out_err.write(key + "\n" + traceback.format_exc() + "\n")
            out_err.close()
        if dev.type == "cuda":
            torch.cuda.empty_cache()


def main(name, budget_h=11.0):
    import torch
    n = max(1, torch.cuda.device_count())
    if not torch.cuda.is_available() and on_tpu():
        n = int(os.environ.get("MANGOFS_TPU_WORKERS", "8"))  # one worker per chip (v5e-8)
    print("devices:", n, "suite:", name, "configs:", len(suite(name)), flush=True)
    if find_groups() is None:
        build_groups()
    procs = []
    for g in range(n):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        if not torch.cuda.is_available() and on_tpu():  # independent single-chip processes
            env.update(PJRT_DEVICE="TPU", TPU_VISIBLE_CHIPS=str(g), TPU_PROCESS_BOUNDS="1,1,1",
                       TPU_CHIPS_PER_PROCESS_BOUNDS="1,1,1", TPU_PROCESS_PORT=str(8476 + g),
                       CLOUD_TPU_TASK_ID="0", TPU_PROCESS_ADDRESSES=f"localhost:{8476 + g}",
                       TPU_RUNTIME_METRICS_PORTS=str(8431 + g))
        code = f"import sys; sys.path.insert(0, '{os.path.dirname(__file__)}'); import run; run.worker('{name}', {g}, {n})"
        procs.append(subprocess.Popen([sys.executable, "-c", code], env=env))
    deadline = time.time() + budget_h * 3600
    t_start = time.time()
    while any(p.poll() is None for p in procs):
        if time.time() > deadline:
            print("time budget reached; stopping workers (resume by re-running)", flush=True)
            for p in procs:
                p.terminate()
            break
        time.sleep(30)
    codes = [p.returncode for p in procs]
    print("finished", codes)
    if n > 1 and all(codes) and time.time() - t_start < 600 and not torch.cuda.is_available():
        # one-process-per-chip TPU setup failed at start-up; use one process on chip 0
        print("all workers failed at start-up; retrying with one worker", flush=True)
        code = f"import sys; sys.path.insert(0, '{os.path.dirname(__file__)}'); import run; run.worker('{name}', 0, 1)"
        subprocess.run([sys.executable, "-c", code])


def build_groups():
    """Compute capture groups once (DINOv2 features + pHash + EXIF bursts)."""
    import numpy as np
    import pandas as pd
    import torch
    from data import Cache, build_groups as bg
    from evaluate import embed
    from models import EmbeddingNet
    ca = Cache(find_root(), "crop", groups_path="/nonexistent")
    dev = pick_device()
    m = EmbeddingNet("dinov2_s", "none", True).to(dev)
    _, feats = embed(m, ca, ca.meta.idx.to_numpy(), dev)
    np.save(f"{WORK}/dinov2_feats.npy", feats.numpy().astype(np.float16))
    group, drop, stats = bg(ca.meta, feats)
    g = pd.DataFrame({"idx": ca.meta.idx, "group": group, "drop": drop})
    g.to_csv(f"{WORK}/groups.csv", index=False)
    sizes = g.groupby("group").size()
    stats.update({"n_groups": int(len(sizes)), "max_group": int(sizes.max()),
                  "mean_group": float(sizes.mean()), "n_drop": int(drop.sum())})
    json.dump(stats, open(f"{WORK}/groups_stats.json", "w"), indent=1)
    print("groups:", stats, flush=True)

## 2.11 Export the benchmark as `MangoFS-BD.zip`

In [ ]:
%%writefile src/export.py
"""Package the unified benchmark as a self-contained zip (MangoFS-BD.zip).

Layout of the archive
  MangoFS-BD/
    README.md
    metadata.csv          one row per image: id, file paths, source dataset,
                          original label, canonical cultivar, camera, EXIF time,
                          capture group, drop flag
    splits/unified_fold{0,1,2}.json   base / novel cultivars + image ids
    splits/lodo_{dataset}.json        train / test ids, seen / novel cultivars
    splits/closed_{dataset}.json      capture-group 70/30 train / test ids
    images/crop/<cultivar>/<id>.jpg   fruit-centred 288x288 crops
    images/full/<cultivar>/<id>.jpg   trimmed full frame, 288x288
"""
import json
import os
import zipfile

import cv2
import numpy as np

from data import DATASETS, protocol_split
from finetune import group_holdout

SHORT = {"MangoImageBD": "MIBD", "MangoClassify12": "MC12", "Mangifera2012": "MF12"}

README = """# MangoFS-BD

Unified few-shot benchmark of {n_cult} Bangladeshi mango cultivars ({n_img} images),
assembled from three public datasets (all CC BY 4.0):

* MangoImageBD (Mendeley hp2cdckpdr v2): original images only
* MangoClassify-12 (Kaggle researchersajid/mangoclassify-12-native-mango-dataset-from-bd)
* Mangifera2012 (Mendeley w5jg84txj8)

Please cite the three source datasets.

File names are `<cultivar>_<source>_<index>.jpg`, where source is MIBD (MangoImageBD),
MC12 (MangoClassify-12) or MF12 (Mangifera2012). The folder name is the label too.
Images: `images/crop` holds fruit-centred square crops and `images/full` holds the
trimmed full frame. Both are 288x288 JPEG.
`metadata.csv`: `group` is the capture group (images that must never be split
across train/test or support/query). Rows with `drop == 1` are near-identical
images with conflicting labels and are excluded from every split.

Splits:
* `unified_fold{{k}}`: 3-fold cultivar cross-validation; every cultivar is novel in exactly one fold.
* `lodo_<dataset>`: leave one dataset out (seen and novel cultivars listed).
* `closed_<dataset>`: closed-set split, whole capture groups held out (about 30%).

Per-cultivar image counts (source dataset columns):

{table}
"""


def export(cache, out_path, quality=95):
    """cache: data.Cache with groups attached (in_memory=False is enough)."""
    meta = cache.meta.copy()
    meta["id"] = [f"{c}_{SHORT[d]}_{i:05d}" for c, d, i in zip(meta.cultivar, meta.dataset, meta.idx)]
    meta["crop_path"] = "images/crop/" + meta.cultivar + "/" + meta.id + ".jpg"
    meta["full_path"] = "images/full/" + meta.cultivar + "/" + meta.id + ".jpg"
    full = np.load(os.path.join(cache.root, "full.npy"), mmap_mode="r")
    keep = ["id", "idx", "crop_path", "full_path", "dataset", "raw_label", "cultivar", "camera",
            "exif_time", "phash", "group", "drop"]
    with zipfile.ZipFile(out_path, "w", zipfile.ZIP_STORED) as z:  # JPEGs are already compressed
        root = "MangoFS-BD/"
        z.writestr(root + "metadata.csv", meta[keep].to_csv(index=False))
        for f in range(3):
            tr, te, info = protocol_split(cache.meta, "unified", f)
            z.writestr(root + f"splits/unified_fold{f}.json",
                       json.dumps({**info, "train_idx": tr.tolist(), "test_idx": te.tolist()}))
        for f, ds in enumerate(DATASETS):
            tr, te, info = protocol_split(cache.meta, "lodo", f)
            z.writestr(root + f"splits/lodo_{ds}.json",
                       json.dumps({**info, "train_idx": tr.tolist(), "test_idx": te.tolist()}))
            m = cache.meta[(cache.meta.dataset == ds) & (cache.meta["drop"] == 0)]
            tr, te = group_holdout(m, seed=0)
            z.writestr(root + f"splits/closed_{ds}.json",
                       json.dumps({"train_idx": tr.tolist(), "test_idx": te.tolist()}))
        table = meta.pivot_table(index="cultivar", columns="dataset", values="idx",
                                 aggfunc="count", fill_value=0).to_markdown()
        z.writestr(root + "README.md", README.format(n_cult=meta.cultivar.nunique(),
                                                     n_img=len(meta), table=table))
        enc = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
        for i, r in enumerate(meta.itertuples()):
            for arr, path in ((cache.images[r.idx], r.crop_path), (full[r.idx], r.full_path)):
                _, buf = cv2.imencode(".jpg", np.ascontiguousarray(arr[..., ::-1]), enc)
                z.writestr(root + path, buf.tobytes())
            if i % 2000 == 0:
                print("exported", i, flush=True)
    print("wrote", out_path, os.path.getsize(out_path) / 2 ** 20, "MB")

# 3. The benchmark
Load the image cache and capture groups, and summarise them.

In [ ]:
import pandas as pd, numpy as np
import run
from data import Cache, protocol_split
cache = Cache(run.find_root(), 'crop', run.find_groups())
m = cache.meta
print(len(m), 'images |', m.cultivar.nunique(), 'cultivars |', m.group.nunique(), 'capture groups |',
      int(m['drop'].sum()), 'dropped (label conflicts)')
m.pivot_table(index='cultivar', columns='dataset', values='idx', aggfunc='count', fill_value=0)

In [ ]:
for f in range(3):
    tr, te, info = protocol_split(m, 'unified', f)
    print(f'fold {f}: {len(info["base"])} base / {len(info["novel"])} novel ->', info['novel'])

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for row, ds in zip(axes, ['MangoImageBD', 'MangoClassify12', 'Mangifera2012']):
    ids = rng.choice(m[m.dataset == ds].idx.to_numpy(), 8, replace=False)
    for ax, i in zip(row, ids):
        ax.imshow(cache.images[i]); ax.set_title(m.cultivar.iat[i], fontsize=8); ax.axis('off')
    row[0].text(-30, 144, ds, rotation=90, va='center', ha='right', fontsize=9)
plt.tight_layout(); plt.show()

# 4. One experiment end to end
Train **ours** (ResNet-18, MLP head, KAN metric, margin 0.1) on the base cultivars of fold 0, then evaluate on 600 episodes per shot over the 8 novel cultivars. This takes about 6 minutes on a T4.

In [ ]:
import train as T
dev = torch.device('cuda')
r = T.run_config(cache, run.m('ours', protocol='unified', fold=0, backbone='resnet18'), dev)
pd.DataFrame({k: {'acc %': 100 * v['acc'], '±95%': 100 * v['acc_ci'], 'macro-F1 %': 100 * v['f1']}
              for k, v in r['results']['all'].items()}).T.round(2)

In [ ]:
pd.DataFrame({k: {'AUROC %': 100 * v['auroc'], '±95%': 100 * v['auroc_ci']}
              for k, v in r['results']['open_set'].items()}).T.round(2)

# 5. Full experiment suites
Each call runs one suite on all GPUs. It appends to `results_<suite>_<gpu>.jsonl` and skips finished configs, so a suite can be resumed in a new session. Approximate wall time on T4 ×2:

| suite | configs | time |
|---|---|---|
| `unified_main` | 117 | ~7–9 h |
| `lodo_main` | 48 | ~3–4 h |
| `unified_extra` | 48 | ~3 h |
| `ablations` | 36 | ~2.5 h |
| `closed_set` (needs `lodo_main` checkpoints) | 12 | ~3 h |
| `episode_ft` | 6 | ~1.5 h |
| `efficiency` | 1 | ~10 min |

Kaggle sessions stop after 12 h. Run one suite per session.

In [ ]:
SUITE = 'unified_main'   # change and re-run for each suite
# run.main(SUITE)        # uncomment to launch

# 6. Tables and figures
After collecting all `results*.jsonl` files, run `analysis/aggregate.py` from the repository. It writes `results/tables.md` and `figures/*.png`.

# 7. Export the benchmark zip

In [ ]:
# import export; export.export(cache, '/kaggle/working/MangoFS-BD.zip')